# Module 4: Vectorization & Broadcasting

### From Loops to Lightning — How Neural Networks Compute

This notebook teaches you how to **perform math on entire arrays at once** — the computational foundation of modern AI. By the end you will understand why loops are slow, how broadcasting eliminates dimension mismatches, and how to implement the forward pass of a neural network layer with zero loops.

We build toward one concrete goal: computing **Y = activation(X @ W + b)** — the forward pass through a neural network layer.

## 📋 Summary: the one-paragraph version

**Vectorization** means replacing Python loops with array operations that the CPU can execute in parallel using SIMD (Single Instruction, Multiple Data) instructions — this is 10-100x faster. **Broadcasting** is NumPy's rule system for automatically extending arrays across mismatched dimensions, so you can add a scalar to a matrix or multiply a `(3,1)` array by a `(1,4)` array without writing explicit loops. Together, these enable you to write the **forward pass** of a neural network — `output = activation(inputs @ weights + bias)` — as a single line that processes an entire batch of examples simultaneously, which is exactly how PyTorch and TensorFlow work under the hood.

> ### 🚀 The mental model
>
> **Loop thinking**: "Process one element, then the next, then the next..."
>
> **Vectorized thinking**: "Process the entire dataset as one operation"
>
> This shift is the difference between code that takes hours and code that takes seconds.

## 🗺️ What you will learn, step by step

| Step | What you do | Key concept |
| ---: | --- | --- |
| 0 | Set up the environment | `numpy`, `matplotlib`, `time` |
| 1 | **Element-wise operations** | Add/multiply arrays, speed comparison vs loops |
| 2 | **Scalar broadcasting** | Multiply/add scalar to array, normalizing data |
| 3 | **The dot product** | Mathematical definition, `np.dot()`, connection to attention |
| 4 | **Matrix multiplication** | Shape requirements, `@` operator, common errors |
| 5 | **Broadcasting rules** | The 3 rules that govern dimension stretching |
| 6 | **The artificial neuron** | Weights · inputs + bias → activation → output |
| 7 | **A layer of neurons** | Weight matrix, batch processing, `Y = activation(X @ W + b)` |
| 8 | **Lab: The forward pass** | Implement a neural network layer with zero loops |

### ✅ Prerequisites

- Basic Python (functions, loops)
- **Module 3** (NumPy arrays, `.shape`, indexing)
- No calculus required (we explain activations intuitively)

---
# Step 0 · Setup

## 0.1 Install the libraries

We use **NumPy** for vectorized operations and **matplotlib** for visualization. The dependency list lives in `pyproject.toml`:

```toml
[project]
requires-python = ">=3.10"
dependencies = [
    "numpy>=1.24",
    "matplotlib>=3.7",
    "python-dotenv",
    "jupyterlab",
    "ipykernel",
]
```

**Recommended setup with `uv`** (from the module directory):

```bash
uv sync
uv run jupyter lab
```

If you are on Colab or another hosted environment without `uv`, run the cell below instead.

In [ ]:
# Run this ONLY if you did NOT use `uv sync` (e.g. on Colab)
%pip install -qU "numpy>=1.24" "matplotlib>=3.7" python-dotenv

## 0.2 Load environment variables

We follow the same `.env` pattern from Module 1 to keep any future API keys out of the notebook.

In [ ]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
print("Environment loaded.")

## 0.3 Imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Callable

%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print("All imports succeeded.")

---
# Step 1 · Element-wise Operations

**Element-wise operations** apply the same mathematical operation to every element in an array **simultaneously**. This is the foundation of vectorization.

## 1.1 Basic arithmetic

NumPy overloads Python's arithmetic operators to work on entire arrays:

In [ ]:
# Create two arrays
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print("a:     ", a)
print("b:     ", b)
print()
print("a + b: ", a + b)
print("a - b: ", a - b)
print("a * b: ", a * b)  # Element-wise multiplication (NOT dot product!)
print("a / b: ", a / b)
print("a ** 2:", a ** 2)

**Key observation:** No loops required. NumPy applies the operation to all elements at once.

## 1.2 Mathematical functions

NumPy provides vectorized versions of math functions:

In [ ]:
x = np.array([0, np.pi/4, np.pi/2, np.pi])

print("x:       ", x)
print("sin(x):  ", np.sin(x))
print("cos(x):  ", np.cos(x))
print("exp(x):  ", np.exp(x))
print("log(x+1):", np.log(x + 1))  # Natural log
print("sqrt(x): ", np.sqrt(x))

## 1.3 Speed comparison: loop vs vectorized

Let's measure the performance difference between a Python loop and a vectorized operation.

In [ ]:
# Create large arrays
size = 1_000_000
a = np.random.rand(size)
b = np.random.rand(size)

# Method 1: Python loop
start = time.perf_counter()
result_loop = []
for i in range(size):
    result_loop.append(a[i] * b[i])
loop_time = time.perf_counter() - start

# Method 2: Vectorized
start = time.perf_counter()
result_vectorized = a * b
vectorized_time = time.perf_counter() - start

print(f"Loop time:       {loop_time*1000:.2f} ms")
print(f"Vectorized time: {vectorized_time*1000:.2f} ms")
print(f"\nSpeedup: {loop_time / vectorized_time:.1f}x faster")
print(f"Results match: {np.allclose(result_loop, result_vectorized)}")

**Visualize the speed difference:**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

methods = ['Python Loop', 'NumPy Vectorized']
times = [loop_time * 1000, vectorized_time * 1000]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, times, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Time (milliseconds)', fontsize=12, fontweight='bold')
ax.set_title('Speed Comparison: Element-wise Multiplication\n(1 million elements)', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0, max(times) * 1.2)

# Add value labels on bars
for bar, time_val in zip(bars, times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{time_val:.2f} ms',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n🚀 Vectorization is {loop_time / vectorized_time:.1f}x faster!")

> ### 💡 Key Insight
>
> **Never write explicit loops for array operations.** If you find yourself writing `for i in range(len(array))`, stop and ask: "Can I vectorize this?" The answer is almost always yes, and the speedup is dramatic.

---
# Step 2 · Scalar Broadcasting

**Broadcasting** is NumPy's mechanism for performing operations on arrays of different shapes. The simplest case: **scalar broadcasting** — applying a single value to every element of an array.

## 2.1 Basic scalar broadcasting

In [ ]:
arr = np.array([1, 2, 3, 4, 5])

print("Original array:", arr)
print("arr + 10:      ", arr + 10)    # Add 10 to every element
print("arr * 2:       ", arr * 2)     # Multiply every element by 2
print("arr / 2:       ", arr / 2)     # Divide every element by 2
print("arr ** 0.5:    ", arr ** 0.5)  # Square root of every element

**What happened?** NumPy automatically "stretched" the scalar across the array:

```
[1, 2, 3, 4, 5] + 10
  ↓
[1, 2, 3, 4, 5] + [10, 10, 10, 10, 10]
  ↓
[11, 12, 13, 14, 15]
```

## 2.2 Practical example: normalizing data

A common operation in machine learning is **z-score normalization**: subtract the mean, divide by standard deviation.

In [ ]:
# Create sample data (e.g., test scores)
scores = np.array([65, 70, 75, 80, 85, 90, 95])

print("Original scores:", scores)
print(f"Mean: {np.mean(scores):.2f}")
print(f"Std:  {np.std(scores):.2f}")

# Normalize (all vectorized, no loops!)
normalized = (scores - np.mean(scores)) / np.std(scores)

print("\nNormalized scores:", normalized)
print(f"New mean: {np.mean(normalized):.2e}")  # Should be ~0
print(f"New std:  {np.std(normalized):.2f}")   # Should be 1

**Visualize the transformation:**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Original scores
ax1.bar(range(len(scores)), scores, color='#3498db', alpha=0.7, edgecolor='black')
ax1.axhline(np.mean(scores), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(scores):.1f}')
ax1.set_title('Original Scores', fontsize=12, fontweight='bold')
ax1.set_xlabel('Student')
ax1.set_ylabel('Score')
ax1.legend()
ax1.grid(alpha=0.3)

# Normalized scores
ax2.bar(range(len(normalized)), normalized, color='#2ecc71', alpha=0.7, edgecolor='black')
ax2.axhline(0, color='red', linestyle='--', linewidth=2, label='Mean = 0')
ax2.set_title('Normalized Scores (z-score)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Student')
ax2.set_ylabel('Standard deviations from mean')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

> ### 💡 Key Insight
>
> **Broadcasting eliminates the need for explicit expansion.** You do not need to manually copy the scalar to match array dimensions — NumPy does it automatically. This is both faster (no memory copies) and cleaner (less code).

---
# Step 3 · The Dot Product

The **dot product** is the fundamental operation in linear algebra and neural networks. It combines multiplication and summation.

## 3.1 Mathematical definition

For two vectors **a** = `[a₁, a₂, a₃]` and **b** = `[b₁, b₂, b₃]`:

$$
\mathbf{a} \cdot \mathbf{b} = a_1 b_1 + a_2 b_2 + a_3 b_3 = \sum_{i=1}^{n} a_i b_i
$$

**In words:** Multiply corresponding elements, then sum the results.

## 3.2 Computing with NumPy

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Method 1: np.dot()
dot_product = np.dot(a, b)
print("np.dot(a, b):", dot_product)

# Method 2: @ operator (Python 3.5+)
dot_product_2 = a @ b
print("a @ b:       ", dot_product_2)

# Manual calculation (for understanding)
manual = a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
print("Manual:      ", manual)

# Vectorized manual calculation
vectorized_manual = np.sum(a * b)
print("np.sum(a*b): ", vectorized_manual)

## 3.3 Geometric meaning

The dot product measures **similarity** between vectors:
- **Positive** → vectors point in similar directions
- **Zero** → vectors are perpendicular (orthogonal)
- **Negative** → vectors point in opposite directions

In [ ]:
# Three pairs of vectors
v1 = np.array([1, 0])
v2_parallel = np.array([2, 0])        # Same direction
v2_perpendicular = np.array([0, 1])   # 90 degrees
v2_opposite = np.array([-1, 0])       # Opposite direction

print("v1 · v2_parallel:     ", np.dot(v1, v2_parallel))      # Positive
print("v1 · v2_perpendicular:", np.dot(v1, v2_perpendicular)) # Zero
print("v1 · v2_opposite:     ", np.dot(v1, v2_opposite))      # Negative

## 3.4 Connection to attention mechanisms

In Transformers, **attention scores** are computed using dot products:

```python
attention_score = query @ key.T  # How similar is the query to this key?
```

Higher dot product = higher attention weight = more relevant.

> ### 💡 Key Insight
>
> **The dot product is the building block of AI.** Neural networks, attention, similarity search — all use dot products. Understanding it deeply is essential.

---
# Step 4 · Matrix Multiplication

**Matrix multiplication** extends the dot product to 2D arrays. This is the operation at the heart of neural networks.

## 4.1 Shape requirements

For matrices **A** (shape `m × n`) and **B** (shape `n × p`):

```
   A      @      B      =      C
(m, n)       (n, p)         (m, p)
```

**Rule:** The **inner dimensions must match** (`n`), and the result has the **outer dimensions** (`m × p`).

```mermaid
graph LR
    A["Matrix A<br/>(m, n)"] --> C["Result C<br/>(m, p)"]
    B["Matrix B<br/>(n, p)"] --> C
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#2ecc71,stroke:#333,stroke-width:2px,color:#fff
```

## 4.2 Basic example

In [ ]:
# Create two matrices
A = np.array([
    [1, 2, 3],
    [4, 5, 6]
])  # Shape: (2, 3)

B = np.array([
    [7, 8],
    [9, 10],
    [11, 12]
])  # Shape: (3, 2)

print("A shape:", A.shape)
print("B shape:", B.shape)

# Matrix multiplication
C = A @ B  # or np.dot(A, B) or np.matmul(A, B)

print("\nC = A @ B:")
print(C)
print("C shape:", C.shape)  # (2, 2)

**How it works:**

Each element `C[i, j]` is the dot product of row `i` from A and column `j` from B:

```
C[0, 0] = A[0, :] · B[:, 0] = [1,2,3] · [7,9,11] = 1*7 + 2*9 + 3*11 = 58
```

## 4.3 Common errors

In [ ]:
# Error 1: Shape mismatch
A = np.array([[1, 2], [3, 4]])      # (2, 2)
B = np.array([[5, 6, 7], [8, 9, 10]]) # (2, 3)

try:
    result = A @ B
    print("This works! Result shape:", result.shape)
except ValueError as e:
    print(f"Error: {e}")

# Try the reverse (will fail)
try:
    result = B @ A  # (2, 3) @ (2, 2) — inner dims don't match!
except ValueError as e:
    print(f"\nError with B @ A: Inner dimensions don't match")
    print(f"B.shape={B.shape}, A.shape={A.shape}")
    print(f"Inner dims: {B.shape[1]} ≠ {A.shape[0]}")

## 4.4 Matrix multiplication is NOT element-wise

**Important distinction:**

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A * B (element-wise):")
print(A * B)  # Multiply corresponding elements

print("\nA @ B (matrix multiplication):")
print(A @ B)  # Dot products of rows and columns

> ### ⚠️ Caution: Always check shapes
>
> Before matrix multiplication, **print the shapes**:
> ```python
> print(f"A: {A.shape}, B: {B.shape}")
> ```
> Ask: "Do the inner dimensions match?" If not, transpose one matrix.

---
# Step 5 · Broadcasting Rules

**Broadcasting** is NumPy's system for handling operations on arrays of different shapes. It follows three simple rules.

## 5.1 The three broadcasting rules

When operating on two arrays, NumPy compares their shapes **element-wise from right to left**. Two dimensions are **compatible** when:

1. **They are equal**, or
2. **One of them is 1**

If these conditions are not met, NumPy raises a `ValueError: operands could not be broadcast together`.

**Rule details:**
- If arrays have different numbers of dimensions, **prepend 1s** to the shape of the smaller array
- Arrays with dimension size **1 are stretched** to match the other array
- If dimensions don't match and neither is 1, **broadcasting fails**

## 5.2 Examples: Compatible shapes

In [ ]:
# Example 1: Scalar + Array
a = np.array([1, 2, 3])  # Shape: (3,)
b = 10                    # Scalar (shape: ())
result = a + b
print("Scalar + Array:")
print(f"  {a} + {b} = {result}")
print(f"  Shapes: {a.shape} + () → {result.shape}\n")

# Example 2: (3, 1) + (3,)
a = np.array([[1], [2], [3]])  # Shape: (3, 1) — column vector
b = np.array([10, 20, 30])     # Shape: (3,) — row-like
result = a + b
print("Column + Row:")
print(f"  Shape {a.shape} + {b.shape} → {result.shape}")
print(f"  Result:\n{result}\n")

# Example 3: (4, 1) + (1, 3)
a = np.array([[1], [2], [3], [4]])  # (4, 1)
b = np.array([[10, 20, 30]])        # (1, 3)
result = a + b
print("Outer broadcasting:")
print(f"  Shape {a.shape} + {b.shape} → {result.shape}")
print(f"  Result:\n{result}")

## 5.3 Visual example: outer product pattern

When you broadcast `(m, 1)` + `(1, n)`, you get `(m, n)` — an **outer product** pattern.

In [ ]:
# Create grid coordinates
x = np.array([[1], [2], [3], [4]])  # (4, 1) — vertical
y = np.array([[10, 20, 30]])        # (1, 3) — horizontal

# Broadcasting creates a grid
grid = x + y

print("Broadcasting (4,1) + (1,3):")
print(grid)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# Input x
axes[0].imshow(x, cmap='Blues', aspect='auto')
axes[0].set_title('x: (4, 1)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('rows')

# Input y
axes[1].imshow(y, cmap='Reds', aspect='auto')
axes[1].set_title('y: (1, 3)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('columns')

# Result
im = axes[2].imshow(grid, cmap='Greens', aspect='auto')
axes[2].set_title('x + y: (4, 3)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('columns')
axes[2].set_ylabel('rows')

# Add colorbar
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 5.4 Incompatible shapes

In [ ]:
# This will fail: (3,) + (4,)
a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])

try:
    result = a + b
except ValueError as e:
    print("Error: Arrays could not be broadcast together")
    print(f"  a.shape: {a.shape}")
    print(f"  b.shape: {b.shape}")
    print(f"  Dimensions 3 and 4 are incompatible (neither is 1)")

> ### 💡 Key Insight
>
> **Broadcasting eliminates dimension-matching boilerplate.** Instead of manually tiling or repeating arrays to match shapes, NumPy does it automatically. This is both faster (no memory copies) and cleaner (less code).

---
# Step 6 · The Artificial Neuron

Now we put everything together to build the **fundamental unit of a neural network**: the artificial neuron.

## 6.1 The neuron equation

A single neuron computes:

$$
\text{output} = \text{activation}\left(\sum_{i=1}^{n} w_i x_i + b\right)
$$

**In NumPy notation:**

```python
z = np.dot(weights, inputs) + bias
output = activation(z)
```

```mermaid
graph LR
    A["Inputs<br/>x₁, x₂, x₃"] --> B["Weighted Sum<br/>w₁x₁ + w₂x₂ + w₃x₃"]
    W["Weights<br/>w₁, w₂, w₃"] --> B
    B --> C["+bias<br/>z = Σ(wᵢxᵢ) + b"]
    C --> D["Activation<br/>f(z)"]
    D --> E["Output<br/>y"]
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style W fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#f39c12,stroke:#333,stroke-width:2px,color:#fff
    style D fill:#16a085,stroke:#333,stroke-width:2px,color:#fff
    style E fill:#2ecc71,stroke:#333,stroke-width:2px,color:#fff
```

## 6.2 Activation functions

**Activation functions** introduce non-linearity. Without them, stacking layers would be pointless (multiple linear transformations = one linear transformation).

### ReLU (Rectified Linear Unit)

The most popular activation function:

$$
\text{ReLU}(x) = \max(0, x) = \begin{cases} x & \text{if } x > 0 \\ 0 & \text{if } x \leq 0 \end{cases}
$$

In [ ]:
def relu(x: np.ndarray) -> np.ndarray:
    """ReLU activation: max(0, x)"""
    return np.maximum(0, x)

# Visualize ReLU
x = np.linspace(-5, 5, 100)
y = relu(x)

plt.figure(figsize=(8, 5))
plt.plot(x, y, linewidth=2.5, color='#2ecc71')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.3)
plt.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.3)
plt.grid(alpha=0.3)
plt.title('ReLU Activation Function', fontsize=14, fontweight='bold')
plt.xlabel('Input (z)', fontsize=12)
plt.ylabel('Output ReLU(z)', fontsize=12)
plt.tight_layout()
plt.show()

print("ReLU properties:")
print("  • Output: [0, ∞)")
print("  • Zeros out negative values")
print("  • Passes positive values unchanged")
print("  • Fast to compute: just max(0, x)")

## 6.3 Implementing a single neuron

In [ ]:
def neuron_forward(inputs: np.ndarray, weights: np.ndarray, bias: float, 
                   activation: Callable = relu) -> float:
    """
    Forward pass through a single neuron.
    
    Args:
        inputs: Input vector (n,)
        weights: Weight vector (n,)
        bias: Bias scalar
        activation: Activation function
        
    Returns:
        Scalar output
    """
    # Weighted sum
    z = np.dot(weights, inputs) + bias
    
    # Activation
    output = activation(z)
    
    return output

# Test with example data
inputs = np.array([1.0, 2.0, 3.0])    # 3 input features
weights = np.array([0.5, -0.3, 0.8])  # Learned weights
bias = 0.1                             # Learned bias

output = neuron_forward(inputs, weights, bias, relu)

print("Single Neuron Example:")
print(f"  Inputs:  {inputs}")
print(f"  Weights: {weights}")
print(f"  Bias:    {bias}")
print(f"\n  z = w·x + b = {np.dot(weights, inputs) + bias:.4f}")
print(f"  output = ReLU(z) = {output:.4f}")

> ### 💡 Key Insight
>
> **A neuron is just a weighted sum + activation.** All the complexity of neural networks comes from:
> 1. **Stacking** many neurons in layers
> 2. **Learning** the weights through backpropagation
> 3. **Depth** — stacking many layers

---
# Step 7 · A Layer of Neurons

A **layer** is a collection of neurons that all receive the same inputs. Instead of computing one neuron at a time, we compute **all neurons simultaneously** using matrix multiplication.

## 7.1 From one neuron to many

**One neuron:**
```python
output = activation(weights · inputs + bias)  # Scalar output
```

**Layer of neurons:**
```python
outputs = activation(inputs @ weights + biases)  # Vector output
```

```mermaid
graph LR
    A["Input Layer<br/>x₁, x₂, x₃"] --> B1["Neuron 1"]
    A --> B2["Neuron 2"]
    A --> B3["Neuron 3"]
    A --> B4["Neuron 4"]
    A --> B5["Neuron 5"]
    B1 --> C["Output Layer<br/>y₁, y₂, y₃, y₄, y₅"]
    B2 --> C
    B3 --> C
    B4 --> C
    B5 --> C
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style B1 fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style B2 fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style B3 fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style B4 fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style B5 fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#2ecc71,stroke:#333,stroke-width:2px,color:#fff
```

## 7.2 The weight matrix

Instead of one weight vector per neuron, we stack them into a **weight matrix**:

```
W = [
    [w₁₁, w₁₂, w₁₃, w₁₄, w₁₅],  ← weights for neuron 1
    [w₂₁, w₂₂, w₂₃, w₂₄, w₂₅],  ← weights for neuron 2
    [w₃₁, w₃₂, w₃₃, w₃₄, w₃₅]   ← weights for neuron 3
]
```

**Shape:** `(input_dim, output_dim)` = `(3, 5)`

## 7.3 Batch processing

In practice, we process **multiple examples at once** (a batch):

```python
X: (batch_size, input_dim)   # Multiple input examples
W: (input_dim, output_dim)   # Weight matrix
b: (output_dim,)             # Bias vector

Z = X @ W + b                # Shape: (batch_size, output_dim)
Y = activation(Z)            # Apply activation element-wise
```

```mermaid
graph LR
    A["Input X<br/>(4, 3)"] --> B["@ Weights W<br/>(3, 5)"]
    B --> C["Z<br/>(4, 5)"]
    D["+ bias b<br/>(5,)"] --> C
    C --> E["ReLU"]
    E --> F["Output Y<br/>(4, 5)"]
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
    style D fill:#f39c12,stroke:#333,stroke-width:2px,color:#fff
    style E fill:#16a085,stroke:#333,stroke-width:2px,color:#fff
    style F fill:#2ecc71,stroke:#333,stroke-width:2px,color:#fff
```

## 7.4 Shape analysis

In [ ]:
# Define dimensions
batch_size = 4    # Process 4 examples at once
input_dim = 3     # Each example has 3 features
output_dim = 5    # Layer has 5 neurons

# Create example data
X = np.random.randn(batch_size, input_dim)  # (4, 3)
W = np.random.randn(input_dim, output_dim)  # (3, 5)
b = np.random.randn(output_dim)             # (5,)

print("Shape analysis:")
print(f"  X: {X.shape} — {batch_size} examples, {input_dim} features each")
print(f"  W: {W.shape} — {input_dim} inputs → {output_dim} outputs")
print(f"  b: {b.shape} — one bias per output neuron")

# Forward pass
Z = X @ W + b     # Broadcasting adds bias to each row
Y = relu(Z)

print(f"\n  Z = X @ W + b: {Z.shape}")
print(f"  Y = ReLU(Z):   {Y.shape}")
print(f"\n  Output: {batch_size} examples × {output_dim} neurons = {Y.shape}")

## 7.5 Implementing a layer

In [ ]:
def layer_forward(X: np.ndarray, W: np.ndarray, b: np.ndarray, 
                  activation: Callable = relu) -> np.ndarray:
    """
    Forward pass through a dense layer.
    
    Args:
        X: Input batch (batch_size, input_dim)
        W: Weight matrix (input_dim, output_dim)
        b: Bias vector (output_dim,)
        activation: Activation function
        
    Returns:
        Output batch (batch_size, output_dim)
    """
    # Linear transformation
    Z = X @ W + b
    
    # Activation
    Y = activation(Z)
    
    return Y

# Test
output = layer_forward(X, W, b, relu)

print("Layer Forward Pass:")
print(f"  Input shape:  {X.shape}")
print(f"  Output shape: {output.shape}")
print(f"\n  First example output: {output[0]}")

> ### 💡 Key Insight
>
> **This is how PyTorch and TensorFlow work.** The `nn.Linear` layer in PyTorch does exactly this:
> ```python
> output = activation(input @ weight.T + bias)
> ```
> (Note: PyTorch uses `weight.T` because it stores weights transposed.)

---
# Step 8 · Lab: The Forward Pass

**Your turn.** Implement a complete forward pass through a neural network layer, then compare its speed to a loop-based implementation.

## 8.1 Task: Implement the forward pass

Given:
- Input: `(4, 3)` — 4 examples, 3 features each
- Weights: `(3, 5)` — 3 inputs → 5 outputs
- Bias: `(5,)` — one per output neuron

Compute: `Y = ReLU(X @ W + b)`

In [ ]:
# Create example data
np.random.seed(42)
X = np.random.randn(4, 3)  # 4 examples, 3 features
W = np.random.randn(3, 5)  # 3 inputs → 5 outputs
b = np.random.randn(5)     # 5 biases

print("Input data:")
print(f"  X.shape: {X.shape}")
print(f"  W.shape: {W.shape}")
print(f"  b.shape: {b.shape}")

# TODO: Implement the forward pass (ZERO LOOPS!)
# Your code here:
Y = relu(X @ W + b)

print(f"\nOutput:")
print(f"  Y.shape: {Y.shape}")
print(f"  Y:\n{Y}")

## 8.2 Challenge: Compare with loop implementation

Implement the same computation using nested loops, then time both versions.

In [ ]:
def forward_pass_loops(X: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Forward pass using nested Python loops (SLOW!).
    """
    batch_size, input_dim = X.shape
    output_dim = W.shape[1]
    
    # Allocate output
    Y = np.zeros((batch_size, output_dim))
    
    # Triple nested loop (horrible!)
    for i in range(batch_size):           # For each example
        for j in range(output_dim):       # For each output neuron
            z = b[j]                      # Start with bias
            for k in range(input_dim):    # Compute weighted sum
                z += X[i, k] * W[k, j]
            Y[i, j] = max(0, z)           # ReLU
    
    return Y

def forward_pass_vectorized(X: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Forward pass using vectorized operations (FAST!).
    """
    return relu(X @ W + b)

# Time both versions
num_runs = 100

# Loop version
start = time.perf_counter()
for _ in range(num_runs):
    Y_loop = forward_pass_loops(X, W, b)
loop_time = time.perf_counter() - start

# Vectorized version
start = time.perf_counter()
for _ in range(num_runs):
    Y_vec = forward_pass_vectorized(X, W, b)
vec_time = time.perf_counter() - start

print(f"Timing ({num_runs} runs):")
print(f"  Loop-based:   {loop_time*1000:.2f} ms")
print(f"  Vectorized:   {vec_time*1000:.2f} ms")
print(f"\n  Speedup: {loop_time / vec_time:.1f}x faster")
print(f"  Results match: {np.allclose(Y_loop, Y_vec)}")

**Visualize the speed difference:**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Triple\nNested Loop', 'Vectorized\n(X @ W + b)']
times = [loop_time * 1000, vec_time * 1000]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, times, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('Time (milliseconds)', fontsize=13, fontweight='bold')
ax.set_title(f'Forward Pass Speed Comparison\n({num_runs} iterations, batch_size=4, 3→5 neurons)', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0, max(times) * 1.3)

# Add value labels
for bar, time_val in zip(bars, times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{time_val:.2f} ms',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Add speedup annotation
speedup = loop_time / vec_time
ax.text(0.5, max(times) * 1.15, f'{speedup:.1f}x faster',
        ha='center', fontsize=16, fontweight='bold', color='#2ecc71',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#2ecc71', linewidth=2))

plt.tight_layout()
plt.show()

## 8.3 Scaling experiment

How does the speedup change with larger batches?

In [ ]:
batch_sizes = [4, 16, 64, 256]
speedups = []

for batch_size in batch_sizes:
    X_test = np.random.randn(batch_size, 3)
    
    # Loop version
    start = time.perf_counter()
    _ = forward_pass_loops(X_test, W, b)
    loop_t = time.perf_counter() - start
    
    # Vectorized version
    start = time.perf_counter()
    _ = forward_pass_vectorized(X_test, W, b)
    vec_t = time.perf_counter() - start
    
    speedup = loop_t / vec_t
    speedups.append(speedup)
    print(f"Batch size {batch_size:3d}: {speedup:5.1f}x faster")

# Plot scaling
plt.figure(figsize=(8, 5))
plt.plot(batch_sizes, speedups, marker='o', markersize=10, linewidth=2.5, color='#3498db')
plt.xlabel('Batch Size', fontsize=12, fontweight='bold')
plt.ylabel('Speedup Factor', fontsize=12, fontweight='bold')
plt.title('Vectorization Speedup vs Batch Size', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> ### 🚀 Key Observation
>
> **The speedup grows with batch size.** Vectorized operations have fixed overhead but scale extremely well. This is why modern ML training uses large batches (32-256 examples) — it maximizes GPU/CPU throughput.

## 8.4 Bonus: Multi-layer network

Stack multiple layers to build a simple neural network:

In [ ]:
def multi_layer_forward(X: np.ndarray, 
                        W1: np.ndarray, b1: np.ndarray,
                        W2: np.ndarray, b2: np.ndarray) -> np.ndarray:
    """
    Two-layer neural network: X → Layer1 → ReLU → Layer2 → ReLU → Output
    """
    # Layer 1
    H1 = relu(X @ W1 + b1)
    
    # Layer 2
    Y = relu(H1 @ W2 + b2)
    
    return Y

# Create a 3 → 8 → 5 network
W1 = np.random.randn(3, 8)   # First layer: 3 inputs → 8 hidden
b1 = np.random.randn(8)
W2 = np.random.randn(8, 5)   # Second layer: 8 hidden → 5 outputs
b2 = np.random.randn(5)

# Forward pass
X_test = np.random.randn(4, 3)  # 4 examples
output = multi_layer_forward(X_test, W1, b1, W2, b2)

print("Two-Layer Network:")
print(f"  Architecture: 3 → 8 → 5")
print(f"  Input shape:  {X_test.shape}")
print(f"  Output shape: {output.shape}")
print(f"\n  Output:\n{output}")

---
# Step 9 · Summary & Key Takeaways

You learned:

1. **Element-wise operations** — Add/multiply entire arrays at once, 50-100x faster than loops
2. **Scalar broadcasting** — NumPy stretches scalars across arrays automatically
3. **The dot product** — `np.dot()` or `@`, measures vector similarity, foundation of attention
4. **Matrix multiplication** — `(m,n) @ (n,p) → (m,p)`, inner dims must match
5. **Broadcasting rules** — Dimensions match if equal or one is 1, automatic stretching
6. **The artificial neuron** — `output = activation(weights · inputs + bias)`
7. **A layer of neurons** — Weight matrix, batch processing, `Y = activation(X @ W + b)`
8. **The forward pass** — Implemented with zero loops, 10-100x faster than loop version

> ### 🧮 The one takeaway
>
> **Never write explicit loops for array operations.** Vectorized code is:
> - **Faster** — 10-100x speedup from CPU vectorization
> - **Cleaner** — Expresses operations at array level, not element level
> - **Correct** — Less opportunity for off-by-one errors and index bugs
>
> The formula `Y = activation(X @ W + b)` is how **all neural networks compute forward passes** — PyTorch, TensorFlow, JAX all do this under the hood.

## What's next?

- **Module 5: DataFrames & Series** — Pandas for structured data manipulation
- **Module 6: Cleaning & Transformation** — Real-world data preprocessing pipelines
- **Module 7: Images as Data** — Computer vision preprocessing, convolutions

## Try it yourself

1. **Implement sigmoid activation** — `σ(x) = 1 / (1 + e^(-x))`, compare to ReLU
2. **Add a third layer** — Extend the multi-layer network to 3 → 8 → 8 → 5
3. **Batch normalization** — Normalize activations: `(Z - mean) / std` before ReLU
4. **Compare with PyTorch** — Install PyTorch, implement the same layer with `nn.Linear`, verify outputs match
5. **Visualize activations** — Plot the distribution of neuron outputs before/after ReLU
6. **Memory analysis** — Use `sys.getsizeof()` to compare memory usage of loop vs vectorized code

Happy vectorizing! 🚀